---
# **機械システム制御特論　課題（5）**
## **リカッチ行列方程式の解より最適レギュレータを設計**
---

### 1. システムおよび重み係数の定義
システムの係数行列 $\bm{A}$ ，$\bm{B}$ ，$\bm{C}$ と重み係数 $\bm{Q}$ ，$\bm{R}$ を定義する．その中で必要なパラメータがあれば，適宜定義する．

In [10]:
import numpy as np

M = 0.7
m = 0.12
J = 9 * 10**-3
l = 0.3
g = 9.8

param1 = (m**2 * g * l**2) / ((M + m) * J + M * m * l**2)
param2 = (M + m) * m * g * l / ((M + m) * J + M * m * l**2)
param3 = (J + m * l**2) / ((M + m) * J + M * m * l**2)
param4 = -(m * l) / ((M + m) * J + M * m * l**2)

A = np.array([
    [0, 0, 1, 0], 
    [0, 0, 0, 1],
    [0, param1, 0, 0], 
    [0, param2, 0, 0]
])

B = np.array([
    [0], 
    [0], 
    [param3], 
    [param4]
])

C = np.array([
    [1, 0, 0, 0], 
    [0, 1, 0, 0], 
    [0, 0, 0, 0], 
    [0, 0, 0, 0]
])

Q = np.eye(4) * 10
R = np.eye(1) * 10

print(
    f"A = \n", 
    f"{A}\n", 
)
print(
    f"B = \n",
    f"{B}\n"
)
print(
    f"C = \n",
    f"{C}\n"
)
print(
    f"Q = \n",
    f"{Q}\n"
)
print(
    f"R = \n",
    f"{R}\n"
)

A = 
 [[ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]
 [ 0.          0.85012048  0.          0.        ]
 [ 0.         19.36385542  0.          0.        ]]

B = 
 [[ 0.        ]
 [ 0.        ]
 [ 1.3253012 ]
 [-2.40963855]]

C = 
 [[1 0 0 0]
 [0 1 0 0]
 [0 0 0 0]
 [0 0 0 0]]

Q = 
 [[10.  0.  0.  0.]
 [ 0. 10.  0.  0.]
 [ 0.  0. 10.  0.]
 [ 0.  0.  0. 10.]]

R = 
 [[10.]]



### 2. リカッチ行列方程式の解
以下の行列方程式を満たすような正定値行列 $\bm{P}$ について解く．
$$
\bm{P}\bm{A}
+\bm{A}^{\mathrm{T}}\bm{P}
-\bm{P}\bm{B}\bm{R}^{-1}\bm{B}^{\mathrm{T}}\bm{P}
+\bm{Q}
=\bm{O}
$$
これを解析的に解くのは困難であるため，数値的に解く．
例えば，Scipyの関数`solve_continuous_are()`が使用できる．

In [14]:
from scipy.linalg import solve_continuous_are

P = solve_continuous_are(A, B, Q, R)

print(
    f"P = \n",
    f"{P}\n"
)

P = 
 [[ 20.08781678  57.9542624   15.17601916  12.49681054]
 [ 57.9542624  754.64354776 103.92064995 162.9348265 ]
 [ 15.17601916 103.92064995  25.44591375  22.33169653]
 [ 12.49681054 162.9348265   22.33169653  36.33345198]]



### 3. フィードバック係数行列の最適解
フィードバック係数行列 $\bm{K}$ は以下の式から得られる．

$$
\bm{K}
=
\bm{R}^{-1}\bm{B}^{\mathrm{T}}\bm{P}
$$

得られた $\bm{K}$ を用いることで，状態フィードバック則

$$
\bm{u}(t)
=
-\bm{K}\bm{x}(t)
$$

により閉ループ系を構成することができる．

In [ ]:
K = np.linalg.inv(R) @ B.T @ P

print(
    f"K = \n",
    f"{K}\n"
)

K = 
 [[ -1.         -25.48878772  -2.00878168  -5.79542624]]



### 4. SciPy `solve_continuous_are()` の内部アルゴリズム

#### 4.0 概要

SciPy の `scipy.linalg.solve_continuous_are()` は，連続時間代数リカッチ方程式（Continuous-time Algebraic Riccati Equation, CARE）

$$
\mathbf{A}^{\mathrm T}\mathbf{P}
+\mathbf{P}\mathbf{A}
-\mathbf{P}\mathbf{B}\mathbf{R}^{-1}\mathbf{B}^{\mathrm T}\mathbf{P}
+\mathbf{Q}
=\mathbf{0}
$$

を数値的に解く関数である．

この関数は**反復法（Newton法やKleinman法など）を用いているわけではなく**，
Hamilton行列に対する**Schur法（Schur Vector Method）**によって直接解を求める．

このアルゴリズムは Laub (1979) により提案され，その後 LAPACK を利用した高精度な実装として SciPy や MATLAB の CARE ソルバに採用されている．

---

#### 4.1. Hamilton行列の構築

まず，システム行列

- 状態行列：$\mathbf A$
- 入力行列：$\mathbf B$
- 状態重み：$\mathbf Q$
- 入力重み：$\mathbf R$

から Hamilton 行列

$$
\mathbf H
=
\begin{bmatrix}
\mathbf A &
-\mathbf B\mathbf R^{-1}\mathbf B^{\mathrm T}
\\
-\mathbf Q &
-\mathbf A^{\mathrm T}
\end{bmatrix}
$$

を構築する．

この行列は

$$
2n\times2n
$$

の大きさを持つ．

---

#### 4.2. Hamilton行列の固有値

Hamilton行列の固有値は

$$
(\lambda,-\lambda)
$$

のように対になって現れる性質を持つ．

例えば

```
-4
-2
-1
1
2
4
```

のようになる．

このうち

$$
\mathrm{Re}(\lambda)<0
$$

となる固有値のみを利用する．

これは閉ループ系が安定となる不変部分空間に対応している．

---

#### 4.3. Schur分解

Hamilton行列に対して実Schur分解

$$
\mathbf H
=
\mathbf U
\mathbf T
\mathbf U^{\mathrm T}
$$

を行う．

ここで

- $\mathbf U$：直交行列
- $\mathbf T$：実Schur標準形（準上三角）

である．

Schur分解は固有値計算より数値的に安定であり，
実際には LAPACK のルーチンを利用して計算される．

---

#### 4.4. 安定部分空間の抽出

Schur分解後，

実部が負である固有値に対応する列だけを取り出す．

得られる行列を

$$
\mathbf U_s
=
\begin{bmatrix}
\mathbf U_1
\\
\mathbf U_2
\end{bmatrix}
$$

と分割する．

ここで

- $\mathbf U_1$：上半分
- $\mathbf U_2$：下半分

である．

---

#### 4.5. リカッチ解の構成

リカッチ方程式の解は

$$
\boxed{
\mathbf P
=
\mathbf U_2
\mathbf U_1^{-1}
}
$$

によって得られる．

これは Laub により示された重要な結果である．

最後に

$$
\mathbf P
=
\frac{\mathbf P+\mathbf P^{\mathrm T}}{2}
$$

として数値誤差による非対称性を除去する場合もある．

---

#### 4.6. 数値計算上の特徴

##### 長所

- 非常に高速
- 数値的に安定
- 初期値不要
- 収束判定不要
- 大規模問題にも利用可能
- MATLAB と同様のアルゴリズム

##### 短所

- アルゴリズムが複雑
- Hamilton行列を構築する必要がある
- Schur分解の理解が必要
- 実装が難しい

---

##### 4.7. 他手法との比較

|手法|特徴|初期値|収束判定|
|---|---|---|---|
|Schur法（SciPy）|Hamilton行列のSchur分解|不要|不要|
|Kleinman法|Lyapunov方程式を反復|必要|必要|
|Newton法|二次収束|必要|必要|
|固定点反復|実装容易|必要|必要|
|オイラー法|簡単だが収束保証なし|必要|必要|

---

##### 4.8. 計算手順まとめ

```text
A,B,Q,R
      │
      ▼
Hamilton行列 H を構築
      │
      ▼
実Schur分解
      │
      ▼
安定固有値（Re(λ)<0）を抽出
      │
      ▼
不変部分空間 U を取得
      │
      ▼
U=[U₁
   U₂]
      │
      ▼
P=U₂U₁⁻¹
      │
      ▼
CARE の正定値解
```

---

##### 4.9. 計算量

状態数を $n$ とすると，

- Hamilton行列のサイズ：$2n \times 2n$
- Schur分解の計算量：$O(n^3)$

である．

したがって，本アルゴリズム全体の計算量も

$$
O(n^3)
$$

となる．

---

##### 4.10. 参考文献

1. A. J. Laub, "A Schur Method for Solving Algebraic Riccati Equations," IEEE Transactions on Automatic Control, Vol. AC-24, No. 6, pp. 913–921, 1979.

2. P. Van Dooren, "A Generalized Eigenvalue Approach for Solving Riccati Equations," SIAM Journal on Scientific and Statistical Computing, Vol. 2, No. 2, pp. 121–135, 1981.

3. SciPy Developers, `scipy.linalg.solve_continuous_are`, SciPy Documentation.